### Imports

In [1]:
import json
import os
import sys

import pandas as pd

### Initialization

In [2]:
sys.path.append("../../")
os.chdir("../..")

In [3]:
os.getcwd()

'/home/nicolas/Documentos/UTN/INA/allium-cepa-classifier'

In [10]:
with open("datasets/allium_cepa_full_images_merged/full_annotations.json") as f:
    annotations = json.load(f)

In [11]:
df = pd.DataFrame(annotations["annotations"])
df_images = pd.DataFrame(annotations["images"])
df_images = df_images.set_index("id")["file_name"]

In [13]:
df["division"] = df["attributes"].apply(lambda x: x["division"])
df["mitosis_stage"] = df["attributes"].apply(lambda x: x["mitosis_stage"])
df["file_name"] = df["image_id"].map(df_images)

In [15]:
value_counts = df["mitosis_stage"].value_counts()

print(value_counts)

mitosis_stage
0    22548
Name: count, dtype: int64


### EDA by dataset

#### INA

In [ ]:
# The list of substrings you want to search for in the 'file_name' column
search_values = ["001", "002", "003", "004", "005", "Entrega"]

# Create a regex pattern to find any of the search values. The '|' acts as an OR.
pattern = "|".join(search_values)

# 1. Create a new DataFrame (`df_ina`) that contains only the rows
#    from the original `df` where 'file_name' matches the pattern.
df_ina = df[df["file_name"].str.contains(pattern, na=False)]

# 2. Now, get the count of rows in the new DataFrame.
#    `len()` is a simple way to get the number of rows.
row_count = len(df_ina)

print(f"INA dataset contains {row_count} annotations/cells.")

INA dataset contains 2030 annotations/cells.


#### onion_cell_merged

In [22]:
# The list of substrings you want to search for in the 'file_name' column
search_values = ["A", "B", "C", "D", "E", "F"]

# Create a regex pattern to find any of the search values. The '|' acts as an OR.
pattern = "|".join(search_values)

# 1. Create a new DataFrame (`df_onion`) that contains only the rows
#    from the original `df` where 'file_name' matches the pattern.
df_onion = df[df["file_name"].str.contains(pattern, na=False)]

# 2. Now, get the count of rows in the new DataFrame.
#    `len()` is a simple way to get the number of rows.
row_count = len(df_onion)

print(f"onion_cell_merged dataset contains {row_count} annotations/cells.")

onion_cell_merged dataset contains 20518 annotations/cells.


### Images split in splits for training

In [ ]:
df.to_csv("df.csv", index=False)

In [2]:
import os
import random
import shutil


def split_images(source_dir, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    """
    Splits images from source_dir into train/val/test folders under output_dir.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1."

    random.seed(seed)

    # Create output directories
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split), exist_ok=True)

    # Collect all image files
    valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")
    images = [f for f in os.listdir(source_dir) if f.lower().endswith(valid_exts)]

    random.shuffle(images)

    # Split indices
    n_total = len(images)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = images[:n_train]
    val_files = images[n_train : n_train + n_val]
    test_files = images[n_train + n_val :]

    # Helper to copy images
    def copy_files(file_list, dest_folder):
        for fname in file_list:
            src = os.path.join(source_dir, fname)
            dst = os.path.join(output_dir, dest_folder, fname)
            shutil.copy2(src, dst)

    copy_files(train_files, "train")
    copy_files(val_files, "val")
    copy_files(test_files, "test")

    print(
        f"✅ Done! {n_train} train, {n_val} val, {len(test_files)} test images saved in '{output_dir}'."
    )

In [3]:
# Example usage:
split_images(
    "/home/jcibeira/INA/allium-cepa-classifier/datasets/allium_cepa_full_images_merged/images",
    "/home/jcibeira/INA/allium-cepa-classifier/datasets/allium_cepa_full_images_merged/images",
)

✅ Done! 450 train, 128 val, 66 test images saved in '/home/jcibeira/INA/allium-cepa-classifier/datasets/allium_cepa_full_images_merged/images'.
